# 03 — Paired neural inverse study

**Controls:** normalized plain PINN; normalized Fourier PINN with fixed weights; the same Fourier representation with bounded adaptive weights. The matched raw reciprocal-span representation is an identity check, not a fourth independent architecture claim.

All controls use the same frozen interior observations, known boundary data, paired feature/collocation seeds, optimizer budget and unit-time PDE residual. Final fixed-budget weights are primary. Initial truth is absent from the training inputs and checkpoint rule. A second checkpoint selected on fixed residual/boundary/observed-data loss is saved only as a diagnostic.

**Resume revision:** 18 planned smooth-case trainings, six already completed in your upload, **12 remaining**. This notebook queues missing jobs only, retaining the original 30,000-step settings. The existing six results are included regardless of their accuracy.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "inverse_study").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the entire study ZIP and open this notebook from its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from inverse_study.common import *
from inverse_study import numerics as num
from inverse_study import workflows as flow
RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.2, "savefig.bbox": "tight"})
print("Study folder:", ROOT)
from inverse_study import completion as study
from inverse_study import completion_checks as checks
REDUCED = RESULTS / "reduced_plan"
REDUCED.mkdir(parents=True, exist_ok=True)
study.verify_protocol_lock(require_reference=True)


In [ ]:
from inverse_study.pinn import tensorflow_available, training_config, train_inverse, fourier_features_numpy
print("TensorFlow available:", tensorflow_available())
display(study.compact_linear_plan())
p = problem_spec("smooth2d")
Z = np.random.default_rng(14).uniform(size=(30,3)) * [1,1,p["T"]]
normalized = fourier_features_numpy(Z, p["bounds"], p["T"], seed=1729)
raw = fourier_features_numpy(Z, p["bounds"], p["T"], seed=1729, raw=True)
assert np.allclose(normalized, raw, rtol=1e-12, atol=1e-12)
print("Matched-map maximum absolute difference:", np.max(np.abs(normalized-raw)))

## 1. Installation smoke check

This is a short execution check, not a scientific result. It saves under `results/neural/smoke` and is excluded from publication tables. Missing TensorFlow produces a clear skip; install the training requirements and restart the kernel before running production jobs.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE and tensorflow_available():
    tiny = flow.linear_plan("screen").iloc[[0]].copy()
    smoke = flow.run_pinn_plan(tiny, profile="smoke", max_runs=1,
                              overrides={"epochs": 3, "N_r": 64, "N_b": 32, "batch_r": 32, "batch_b": 16,
                                         "validation_r": 32, "validation_b": 16, "eval_per_dim": 17,
                                         "checkpoint_every": 1, "history_every": 1, "validate_every": 1})
    display(smoke)
else:
    print("Smoke not run. This cell does not start any production experiment.")

## 2. Complete the reduced production cohort

For each of the three methods, retain three clean training seeds and three noisy repetitions. Each noisy repetition has one predetermined noise/training seed pair, shared across methods. The complete cohort has 18 runs.

`MAX_NEW_JOBS=None` processes all pending runs. Set it to `1` to process only the next missing or resumable job; completed jobs do not consume this limit. Leave `CONFIG_OVERRIDES={}` to preserve the settings of your existing results. Signature conflicts are reported before launching work.

Re-running this cell is safe: matching completed jobs are skipped, and interrupted jobs continue from the latest saved checkpoint. A checkpoint interval can leave a small number of unsaved optimizer updates to repeat after an interruption.


In [ ]:
RUN_PRODUCTION = True
MAX_NEW_JOBS = None
CONFIG_OVERRIDES = {}
plan = study.compact_linear_plan()
plan.to_csv(REDUCED / "linear_plan.csv",index=False)
inventory = study.plan_inventory(plan, overrides=CONFIG_OVERRIDES)
display(inventory[["method","noise_level","noise_seed","training_seed","status","reason"]])
print("Already completed:",int(inventory.status.eq("completed").sum()),
      "| pending:",int(inventory.status.isin(["missing","resumable"]).sum()))
if RUN_PRODUCTION:
    completed = study.run_missing(plan, max_new_jobs=MAX_NEW_JOBS, overrides=CONFIG_OVERRIDES)
    display(completed)
else:
    print("Production training is OFF; the inventory above is read-only.")


## 3. Check the matched representation once

The raw reciprocal-span implementation is algebraically canonicalized to the same normalized Fourier computation. With equal underlying draws, initialization, points and losses, its trajectory should match. A difference would indicate an implementation or normalization mismatch; equality does not establish a new architecture advantage.

In [ ]:
RUN_MATCHED_CONTROL = False
if RUN_MATCHED_CONTROL:
    folder = freeze_observations(problem_spec(), 33, .05, 3100)
    p, obs, meta = load_observations(folder)
    p_train = {k:v for k,v in p.items() if k not in ["modes", "amplitudes"]}
    runs = []
    for method in ["fourier_adaptive", "raw_matched"]:
        cfg = training_config("smoke", method=method, training_seed=2026,
                              feature_seed=72026, collocation_seed=82026, residual_scale=p["T"])
        runs.append(train_inverse(p_train, obs, cfg, RESULTS / "identity_checks"))
    a = np.load(runs[0]["predictions_file"])["theta_initial"]
    b = np.load(runs[1]["predictions_file"])["theta_initial"]
    assert np.allclose(a,b,rtol=1e-6,atol=1e-7)
    print("Maximum trajectory-end difference:", np.max(np.abs(a-b)))
else:
    print("Matched-map identity was checked above; short training control is optional.")

## 4. Inspect the full selected cohort

Completed results remain part of the report even when a method performs poorly. Do not choose a different seed based on the true initial-state error. The report in 07 summarizes the three repetitions descriptively and keeps noise and training randomness combined for the noisy cases.

Independent replay refinement below evaluates each completed final checkpoint on three forward grids. It does not retrain the model. Results are saved separately from imported evaluations and reused on subsequent notebook runs.


In [ ]:
inventory = study.plan_inventory(plan)
display(inventory[["method","noise_level","noise_seed","training_seed","status"]])
candidate, blocked = flow.collect_candidates()
if len(blocked): display(blocked)
if len(candidate):
    selected = candidate[candidate.run_folder.isin(inventory.run_folder)]
    useful = [c for c in ["case","method","noise_level","noise_seed","training_seed",
                          "initial_relative_l2","independent_observation_residual",
                          "training_wall_time_s"] if c in selected]
    display(selected[useful])


In [ ]:
RUN_REPLAY_REFINEMENT = True
MAX_NEW_REPLAYS = None
if RUN_REPLAY_REFINEMENT:
    replay_summary = checks.refine_neural_replays(plan, max_new_runs=MAX_NEW_REPLAYS)
    display(replay_summary)
else:
    print("Replay refinement is OFF; completed neural weights are unchanged.")
